## Import Python Libraries and load environment variables

In [1]:
import os
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

/workspaces/hai5016-project/.venv/lib/python3.11/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

## Create llm (model) instance and test with simple question

In [2]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

In [3]:
llm.invoke("What is the capital of South Korea?").content

'Seoul.'

## Create a (langchain) Agent instance and test with simple question

In [4]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'I don’t have real-time exchange rates in this chat. If you want the exact current KRW → USD rate, I can look it up for you if you enable internet access or you can check one of these quick sources:\n- Google: search “KRW to USD”\n- XE, OANDA, Yahoo Finance\n- Your bank’s or a currency app\n\nIf you tell me an amount in KRW, I can convert it to USD once you provide the latest rate. Or I can give a rough estimate using a typical range (for reference, USD is usually around 1,100–1,400 KRW, but this varies daily). Would you like me to fetch the live rate or convert a specific amount using a rate you provide?'}]


## Create a helper function to beautify llm responses in our Jupyter Notebook

In [6]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [7]:
bfmsgs(result)

I don’t have real-time exchange rates in this chat. If you want the exact current KRW → USD rate, I can look it up for you if you enable internet access or you can check one of these quick sources:
- Google: search “KRW to USD”
- XE, OANDA, Yahoo Finance
- Your bank’s or a currency app

If you tell me an amount in KRW, I can convert it to USD once you provide the latest rate. Or I can give a rough estimate using a typical range (for reference, USD is usually around 1,100–1,400 KRW, but this varies daily). Would you like me to fetch the live rate or convert a specific amount using a rate you provide?

## Create a dummy function that returns 1500 as KRW > USD FX

In [8]:
def get_fx(currency: str) -> str:
    """Get exchange rate for a given currency."""
    if currency.upper() in ["KRW", "USD"]:
        return "1500"

In [9]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [10]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's USD worth in KRW right now?"}]}
)
bfmsgs(result)

As of now, 1 USD is about 1,500 KRW. (1 USD ≈ 1,500 KRW)

Rates can fluctuate, so if you want a precise amount for a specific transaction, tell me the USD amount and I’ll convert it.

In [18]:
# @tool # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from div id='wrapper'."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    # Decode HTML bytes to text
    text = raw.decode("utf-8", errors="replace")

    # Return all page text (not only div id="wrapper")
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text(separator=" ", strip=True)
    soup = BeautifulSoup(text, "html.parser")
    wrapper = soup.find("div", id="wrapper")

    if wrapper:
        return wrapper.get_text(strip=True)
    return "Wrapper div not found"

In [19]:
fetch_text_from_url("http://hfspn.co/")

'HUFSPOON MAP # Q & A HUFSPOON FACEBOOK MONDAYSPOON 개인정보처리방침 MAP # Q & A HUFSPOON FACEBOOK MONDAYSPOON 개인정보처리방침 menu THU\xa0 7 / 5 navigate_before navigate_next date_range restaurant Humanities restaurant Faculty restaurant Skylounge translate LANGUAGES Breakfast 08:00~10:00 / 4000 won Doenjang Jigae (Soybean Stew) 685 Kcal 비엔나구이/Ketchup Seasoned Beansprouts Kimchi , Sweet Rice Drink 학생1000 Pork alarm_on favorite_border 44 Lunch 1 11:00~14:30 / 4,000 won Spicy Grilled Chicken With Rice 725 Kcal Tofu Soybean Soup Seasoned Bellflower And Cucumber 환어묵볶음 , Kimchi Chicken alarm_on favorite_border 205 Lunch 2 11:00~14:30 / 3,500 won Seolleongtang (Oxbone Soup) 550 Kcal Seasoned Bellflower And Cucumber Radish Kimchi Beef,Oxbone alarm_on favorite_border 134 Lunch Noodles 11:00~14:00 / 3,000 won Naengmyeon (Noodles With Cold Soup) 520 Kcal alarm_on favorite_border 46 Dinner 16:40~18:40 / 3,500 won Pork Bulgogi With Rice 650 Kcal Beansprout Soup Stir Fried Potato , Kimchi Pork alarm_on favorite_

In [13]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [14]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT
)

- https://www.hanyang.ac.kr/re12
- http://hfspn.co/

In [20]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at university campuses? Find it at https://www.hanyang.ac.kr/re12 and http://hfspn.co/"}]}
)
bfmsgs(result)

Here are today’s menus from the two pages you shared. Date context pulled from the pages (today is 2026-05-07).

1) Hanyang University RE12 (HY-in) page
- Breakfast (08:00–10:00, 4,000 won)
  - Doenjang Jigae (Soybean Stew) – 685 kcal
  - Seasoned Beansprouts
  - Kimchi
  - Sweet Rice Drink
  - Note: Breakfast is listed as 4,000 won for the set
- Lunch 1 (11:00–14:30, 4,000 won)
  - Spicy Grilled Chicken With Rice (725 kcal)
  - Tofu Soybean Soup
  - Seasoned Bellflower and Cucumber
  - Kimchi
- Lunch 2 (11:00–14:30, 3,500 won)
  - Seolleongtang (Oxbone Soup) – 550 kcal
  - Seasoned Bellflower and Cucumber
  - Radish Kimchi
  - Beef, Oxbone
- Lunch Noodles (11:00–14:00, 3,000 won)
  - Naengmyeon (Noodles With Cold Soup) – 520 kcal
- Dinner (16:40–18:40, 3,500 won)
  - Pork Bulgogi With Rice – 650 kcal
  - Beansprout Soup
  - Stir-fried Potato, Kimchi
- Snacks (09:00–18:40)
  - Ramen With Ricecake Balls – 2,400 won
  - Cheese Ramen – 2,400 won
  - Egg Toast – 2,100 won
  - Cheese Toast? (listed in a few lines) – 2,100 won
  - Bowl Of Rice – 700 won
  - Cheese – 400 won
- Lunch set option (11:00–13:30, 6,500 won)
  - Multigrain Rice, plus other components (listed with assorted sides and beef/pork options in the broader page)
  - Note: This is a more elaborate set, priced at 6,500 won

2) HFSPN.co (Mondayspoon page)
- Breakfast (08:00–10:00 range shown on the page; prices around 4,000 won)
  - Doenjang Jigae (Soybean Stew) is listed as a breakfast option with a 4,000 won price in the main “Breakfast” block
  - Seasoned Beansprouts
  - Kimchi
  - Sweet Rice Drink
- Lunch (11:00–14:30 range; main options around 4,000 won)
  - Spicy Grilled Chicken With Rice
  - Tofu Soybean Soup
  - Seasoned Bellflower and Cucumber
  - Kimchi
- Additional Lunch Noodle options (3,000–4,500 won range)
  - Naengmyeon (Noodles with Cold Soup)
- Dinner (16:40–18:40 range; prices around 3,500–4,000 won)
  - Pork Bulgogi With Rice
  - Beansprout Soup
  - Stir-fried Potato, Kimchi
- Snacks (multiple items with prices around 2,000–2,400 won)
  - Ramen With Ricecake Balls
  - Cheese Ramen
  - Egg Toast
  - Ramen
  - Bowl of Rice
  - Cheese
- Lunch set / Brunch options (6,500–7,500 won range)
  - Various multi-dish sets (e.g., Multigrain Rice with beef/pork components)
  - Brunch sets including combinations like pork Bulgogi, beef dishes, seafood options, etc.

Notes
- The HY-in / RE12 page and the HFSPN page both show today’s lines with times and prices, and they include a mix of Korean staples (doenjang jjigae, seolleongtang, bulgogi, naengmyeon) plus snack items (ramen, toast) and larger lunch sets.
- Prices are in Korean won (₩) as listed on the pages.
- If you’d like, I can pull exact item-by-item translations from the HFSPN page or extract today’s menu for a particular campus or dining hall more succinctly (e.g., only main meals, or only lunch items).

Would you like me to filter this to just the main meals for today, or pull menus for a specific campus location?